# Дообучение детектора номеров (прицепы)

Справа: **Accelerator → GPU T4 x2**, **Internet → On** (Settings → Turn on internet: пункт переключает, проверь, что стало «Turn off»), **Add Input** → приватный датасет с архивом `anpr-uz-det.zip`. Затем «Run All».

In [ ]:
import glob, os, shutil

# /kaggle/input только для чтения, а YOLO пишет кэш разметки рядом с ней
SRC = os.path.dirname(glob.glob('/kaggle/input/**/det/data.yaml', recursive=True)[0])
shutil.copytree(SRC, '/kaggle/working/det', dirs_exist_ok=True)
for part in ('train', 'val'):
    print(part, len(os.listdir(f'/kaggle/working/det/images/{part}')), 'фото')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics

In [ ]:
# Дообучение, а не обучение с нуля: старт с детектора, который уже видит
# однострочные номера. Низкий lr0 — чтобы не растерять это умение.
%cd /kaggle/working/det
!yolo detect train model=yolo11n-plate.pt data=data.yaml imgsz=960 epochs=60 \
    batch=16 lr0=0.002 patience=15 device=0 project=/kaggle/working/runs name=det \
    exist_ok=True plots=False

In [ ]:
!ls -la /kaggle/working/runs/det/weights/
!tail -5 /kaggle/working/runs/det/results.csv

Готово: **Output → runs/det/weights/best.pt**. Дома замер на эталоне, размеченном руками:

```
./.venv/bin/python -m tools.eval_detector models/yolo11n-plate.pt путь/к/best.pt
```